# 泰国文化多模态探索平台

基于「万卷·丝路」泰语多模态数据集，在昇思MindSpore框架 + 香橙派AIpro开发板上运行。

**数据来源：**
- SFT数据：泰语文化/生活/数学/代码指令问答
- 图文数据：泰国文化图片 + 泰语描述
- 视频数据：泰语视频元数据，含字幕/摘要/标签

**功能模块：**
1. **多模态RAG问答** — 融合SFT文本知识库 + 图文caption，回答时附带相关图片
2. **图文文化浏览** — 按标签分类浏览泰国文化图片，展示泰语描述
3. **视频内容检索** — 关键词/标签检索泰语视频，展示摘要和跳转链接

**运行环境：** 香橙派AIpro 12G | CANN 8.1RC1 | MindSpore 2.5.0 | MindNLP 0.4.1 | Python 3.9

## 环境准备

In [1]:
!pip install mindnlp==0.4.1
!pip install gradio==4.44.0
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import mindspore
import mindnlp
import sklearn
import gradio
import subprocess
import pkg_resources

print(f'MindSpore: {mindspore.__version__}')
print(f'MindNLP:   {pkg_resources.get_distribution("mindnlp").version}')
print(f'Gradio:    {gradio.__version__}')
print(f'Sklearn:   {sklearn.__version__}')

result = subprocess.run(['npu-smi', 'info'], capture_output=True, text=True)
print('NPU: 正常' if result.returncode == 0 else 'NPU: 未检测到，使用CPU')

[WARNING] ME(9270:246290119323680,MainProcess):2026-05-06-16:12:20.605.30 [mindspore/run_check/_check_version.py:324] MindSpore version 2.5.0 and Ascend AI software package (Ascend Data Center Solution)version 7.7 does not match, the version of software package expect one of ['7.5', '7.6']. Please refer to the match info on: https://www.mindspore.cn/install
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, get

MindSpore: 2.5.0
MindNLP:   0.4.1
Gradio:    4.44.0
Sklearn:   1.4.0
NPU: 正常


## 数据加载与分析

加载三种模态的泰语数据，分析数据分布。

In [ ]:
import json
from collections import Counter

# 只加载 2000 条 SFT 数据，减少内存占用
MAX_SFT = 2000
sft_data = []
with open('data/raw/sft/th/th.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= MAX_SFT:
            break
        line = line.strip()
        if line:
            sft_data.append(json.loads(line))

print(f'SFT数据总条数: {len(sft_data)} (限制 {MAX_SFT} 条)')
type_counter = Counter(d['type'] for d in sft_data)
print('类型分布:')
for t, c in sorted(type_counter.items(), key=lambda x: -x[1]):
    print(f'  {t:<12} {c:>5}条  {chr(9608) * (c // 200)}')

import gc
gc.collect()  
print('内存清理完成')

SFT数据总条数: 2000 (限制 2000 条)
类型分布:
  culture       2000条  ██████████
内存清理完成


In [2]:
# 展示各类型SFT数据示例
print(' SFT数据各类型示例 ')
shown = {}
for d in sft_data:
    t = d['type']
    if t not in shown:
        shown[t] = d
        print(f'\n[{t}]')
        print(f'  问: {d["prompt"].strip()}')
        print(f'  答: {d["completion"].strip()}')
    if len(shown) == 5:
        break

 SFT数据各类型示例 

[culture]
  问: ใครคือกษัตริย์ที่ยิ่งใหญ่ที่สุดในประวัติศาสตร์ไทย?
  答: กษัตริย์ที่ยิ่งใหญ่ที่สุดในประวัติศาสตร์ไทย เช่น รัชกาลที่ 5


In [ ]:
# 只加载 3000 条图文数据，减少内存占用
MAX_IMAGE = 3000
image_data = []
with open('data/raw/image/th/th_image_text_pair.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= MAX_IMAGE:
            break
        line = line.strip()
        if line:
            image_data.append(json.loads(line))

print(f'图文数据总条数: {len(image_data)} (限制 {MAX_IMAGE} 条)')
label_counter = Counter()
for d in image_data:
    for lv2 in d.get('labels', {}).get('pjwk_cates', {}).get('level2', []):
        label_counter[lv2] += 1
print('图片标签分布（level2）:')
for label, cnt in label_counter.most_common():
    print(f'  {label:<20} {cnt:>6}张')

gc.collect()  
print('内存清理完成')

图文数据总条数: 3000 (限制 3000 条)
图片标签分布（level2）:
  食物                     2090张
  乡村场景                    946张
内存清理完成


In [ ]:
# 展示前5条图文数据
print('=== 图文数据样本 ===')
for i, d in enumerate(image_data[:5]):
    caption = d.get('captions', {}).get('content', '')
    img_url = d.get('image', {}).get('path', '')
    lv2 = d.get('labels', {}).get('pjwk_cates', {}).get('level2', [])
    res = d.get('image', {}).get('resolution', [])
    print(f'[{i+1}] 标签: {lv2}  分辨率: {res}')
    print(f'     描述: {caption}')
    print(f'     URL:  {img_url[:80]}')
    print()

=== 图文数据样本 ===
[1] 标签: ['食物']  分辨率: [400, 600]
     描述: น้ำฝน
     URL:  https://www.khaosod.co.th/wpapp/uploads/2023/04/%E0%B8%A7%E0%B8%B1%E0%B8%99%E0%B

[2] 标签: ['食物']  分辨率: [267, 400]
     描述: กล้วยบวชชี
     URL:  https://food.mthai.com/app/uploads/2012/07/กล้วยบวชชี-3.jpg

[3] 标签: ['食物']  分辨率: [494, 800]
     描述: ตานุช ข้าวแกงปักษ์ใต้เสน่ห์ปลายจวักรสดั้งเดิม
     URL:  https://www.khaosod.co.th/wpapp/uploads/2020/03/d1-1.jpg

[4] 标签: ['食物']  分辨率: [332, 500]
     描述: รวมแบบกระทงต่างๆ กระทงกาบกล้วย กระทงดอกบัว กระทงกะลา
     URL:  http://scoop.mthai.com/app/uploads/2014/11/kratong-Lotus3.jpg

[5] 标签: ['食物']  分辨率: [392, 696]
     描述: เปิดใจ
     URL:  https://www.khaosod.co.th/wpapp/uploads/2023/06/20230620_163804-696x392.jpg



In [ ]:
# 加载 th_image_caption.jsonl，图片+详细caption
caption_data = []
with open('data/raw/image/th/th_image_caption.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                caption_data.append(json.loads(line))
            except Exception:
                continue

print(f'image_caption 数据条数: {len(caption_data)}')
# 展示前3条结构
print('\n样本结构如下:')
for d in caption_data[:3]:
    print(json.dumps(d, ensure_ascii=False)[:200])
    print()

image_caption 数据条数: 80035

样本结构如下:
{"img_id": "00177b4c605fa0f20b68b342cf492aef53b8f63005217dd7febf2588aebd2641", "image": {"path": "https://www.khaosod.co.th/wpapp/uploads/2021/05/388230-696x460.jpg", "resolution": [696, 460], "size":

{"img_id": "001eba5de35c4d090d7dfb0a8adb2ab986a1d63db67babbe9e6b6a8a5640c4e5", "image": {"path": "https://www.m2fnews.com/media/content/2022/03/27/372654.jpg", "resolution": [1500, 999], "size": 159.6

{"img_id": "002a95fd97d4ab30957d76a4df1593a5b85556be7f5d1a361650313a03a70d59", "image": {"path": "https://www.khaosod.co.th/wpapp/uploads/2023/01/%E0%B8%9B%E0%B8%A5%E0%B8%81%E0%B8%82%E0%B8%B2%E0%B8%97



In [ ]:
MAX_VIDEOS = 1000  # 减少内存占用

video_data = {}
with open('data/raw/video/th/multilingual_thai_meta_out.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if len(video_data) >= MAX_VIDEOS:
            break
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except Exception:
            continue
        url = obj.get('url', '')
        if not url or url in video_data:
            continue
        summary = ''
        for cap in obj.get('caption', []):
            if cap.get('type') == 'summary':
                summary = cap.get('content', '')
                break
        labels = obj.get('labels', {}).get('pjwk_cates', {})
        video_data[url] = {
            'url': url,
            'summary': summary,
            'level1': labels.get('level1', []),
            'level2': labels.get('level2', []),
        }

video_list = list(video_data.values())
print(f'视频去重后总数: {len(video_list)}（限制 {MAX_VIDEOS} 条）')
v_label_counter = Counter()
for v in video_list:
    for lv2 in v['level2']:
        v_label_counter[lv2] += 1
print('视频标签分布（level2）:')
for label, cnt in v_label_counter.most_common(10):
    print(f'  {label:<20} {cnt:>5}个')

del video_data  
gc.collect()  
print('内存清理完成')

视频去重后总数: 1000（限制 1000 条）
视频标签分布（level2）:
  people                 203个
  unknown                173个
  movie & animation      114个
  howto                   80个
  interview               78个
  game                    67个
  sports                  57个
  technology & military    55个
  news                    42个
  music                   39个
内存清理完成


In [7]:
print(f'视频总数: {len(video_list)}')

# level1 分布
lv1_counter = Counter()
for v in video_list:
    for lv1 in v['level1']:
        lv1_counter[lv1] += 1
print('\n一级标签分布:')
for label, cnt in lv1_counter.most_common():
    bar = chr(9608) * (cnt // 50)
    print(f'  {label:<20} {cnt:>5}个  {bar}')

print('\n二级标签分布（全部）:')
for label, cnt in v_label_counter.most_common():
    bar = chr(9608) * (cnt // 50)
    print(f'  {label:<20} {cnt:>5}个  {bar}')

# 展示几条视频样本
for v in video_list[:3]:
    print(f'URL:    {v["url"]}')
    print(f'标签:   {v["level1"]} / {v["level2"]}')
    print(f'摘要:   {v["summary"][:100]}...')
    print()

视频总数: 1000

一级标签分布:
  people                 306个  ██████
  scenary                285个  █████
  general                236个  ████
  unknown                173个  ███

二级标签分布（全部）:
  people                 203个  ████
  unknown                173个  ███
  movie & animation      114个  ██
  howto                   80个  █
  interview               78个  █
  game                    67个  █
  sports                  57个  █
  technology & military    55个  █
  news                    42个  
  music                   39个  
  voyage                  35个  
  religion                32个  
  animal                  25个  
URL:    https://www.youtube.com/watch?v=--9OlObKEo8
标签:   ['people'] / ['animal']
摘要:   ...

URL:    https://www.youtube.com/watch?v=--_xi0fogSk
标签:   ['unknown'] / ['unknown']
摘要:   ...

URL:    https://www.youtube.com/watch?v=--tAKJhnpfA
标签:   ['scenary'] / ['game']
摘要:   ...



## 构建多模态知识库

将SFT文本问答对 + 图文caption合并为统一知识库，使用TF-IDF字符级n-gram向量化，检索时同时返回文字答案和相关图片。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 只对 SFT 数据做向量化
sft_entries = []
for d in sft_data:
    sft_entries.append({
        'type': 'sft',
        'text': d['prompt'].strip() + ' ' + d['completion'].strip(),
        'prompt': d['prompt'].strip(),
        'answer': d['completion'].strip(),
        'category': d['type'],
    })

# 图文数据只存列表，不向量化，省内存
image_entries = []
for d in image_data:
    caption = d.get('captions', {}).get('content', '').strip()
    img_url = d.get('image', {}).get('path', '')
    if caption and img_url:
        image_entries.append({
            'text': caption,
            'image_url': img_url,
        })

print(f'SFT 条目: {len(sft_entries)}（将向量化）')
print(f'图文条目: {len(image_entries)}（不向量化，实时匹配）')

SFT 条目: 2000（将向量化）
图文条目: 3000（不向量化，实时匹配）


In [ ]:
# 只向量化 SFT 数据
sft_texts = [e['text'] for e in sft_entries]

sft_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 3),
    max_features=3000,  # 减少内存占用
    sublinear_tf=True,
)
sft_matrix = sft_vectorizer.fit_transform(sft_texts)

print(f'SFT 向量化完成: {sft_matrix.shape[0]} 条，{sft_matrix.shape[1]} 维')

del sft_texts  
gc.collect()  
print('内存清理完成')

SFT 向量化完成: 2000 条，3000 维
内存清理完成


In [ ]:
def retrieve_sft(query, top_k=3):
    """SFT 用 TF-IDF 检索"""
    qv = sft_vectorizer.transform([query])
    scores = cosine_similarity(qv, sft_matrix).flatten()
    idx_sorted = np.argsort(scores)[::-1]
    results = []
    for idx in idx_sorted[:top_k * 5]:
        if scores[idx] <= 0:
            break
        e = sft_entries[idx]
        results.append({**e, 'score': float(scores[idx])})
        if len(results) >= top_k:
            break
    return results

def retrieve_images(query, top_k=3):
    # 图文用简单文本匹配，向量化，省内存
    query_lower = query.lower()
    results = []
    for e in image_entries:
        if query_lower in e['text'].lower():
            results.append({**e, 'score': 1.0})
            if len(results) >= top_k:
                break
    return results

# 测试
test_q = 'มวยไทยคืออะไร'
r_sft = retrieve_sft(test_q, top_k=2)
r_img = retrieve_images(test_q, top_k=2)
print(f'SFT 检索: {len(r_sft)} 条')
for r in r_sft:
    print(f'  score={r["score"]:.4f}  {r["prompt"][:50]}')
print(f'图文检索: {len(r_img)} 条')

SFT 检索: 2 条
  score=0.5691  กีฬาประจำชาติของไทยคืออะไร?
  score=0.5691  กีฬาประจำชาติของไทยคืออะไร?
图文检索: 0 条


## 加载wongwian-micro-instruct模型

使用MindNLP加载wongwian-micro-instruct，支持泰语，适合在香橙派开发板上运行。

In [11]:
import subprocess
result = subprocess.run(['npu-smi', 'info'], capture_output=True, text=True)
print(result.stdout)

+--------------------------------------------------------------------------------------------------------+
| npu-smi 25.2.0                                   Version: 25.2.0                                       |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 0       310B1                 | Alarm           | 0.0          66                15    / 15            |
| 0       0                     | NA              | 0            4572 / 11578                            |
+===============================+=================+======================================================+



In [12]:
import mindspore
from mindnlp.transformers import LlamaForCausalLM, LlamaTokenizer
import gc
from mindspore._c_expression import disable_multi_thread

disable_multi_thread()
gc.collect()

model_path = './models/wongwian-micro-instruct'

print('加载 tokenizer...')
tokenizer = LlamaTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.unk_token
print('✓ tokenizer 加载完成')

print('\n加载模型...')
model = LlamaForCausalLM.from_pretrained(
    model_path,
    ms_dtype=mindspore.float16,
    low_cpu_mem_usage=True,
    use_safetensors=False
)
model.set_train(False)
print('✓ 模型加载完成')

[WARNING] ME(9206:246289758904352,MainProcess):2026-05-07-20:47:41.951.409 [mindspore/run_check/_check_version.py:324] MindSpore version 2.5.0 and Ascend AI software package (Ascend Data Center Solution)version 7.7 does not match, the version of software package expect one of ['7.5', '7.6']. Please refer to the match info on: https://www.mindspore.cn/install
[WARNING] ME(9206:246289758904352,MainProcess):2026-05-07-20:47:48.349.431 [mindspore/run_check/_check_version.py:342] MindSpore version 2.5.0 and "te" wheel package version 7.7 does not match. For details, refer to the installation guidelines: https://www.mindspore.cn/install
[WARNING] ME(9206:246289758904352,MainProcess):2026-05-07-20:47:48.359.043 [mindspore/run_check/_check_version.py:349] MindSpore version 2.5.0 and "hccl" wheel package version 7.7 does not match. For details, refer to the installation guidelines: https://www.mindspore.cn/install
[WARNING] ME(9206:246289758904352,MainProcess):2026-05-07-20:47:48.360.941 [minds

加载 tokenizer...


LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`.`PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


✓ tokenizer 加载完成

加载模型...


[WARNING] DEVICE(9206,dfffcd976020,python):2026-05-07-20:48:11.714.887 [mindspore/ccsrc/plugin/device/ascend/hal/device/ascend_memory_adapter.cc:118] Initialize] Free memory size is less than half of total memory size.Device 0 Device MOC total size:12140449792 Device MOC free size:5539405824 may be other processes occupying this card, check as: ps -ef|grep python


✓ 模型加载完成


In [ ]:
# 测试推理（泰语）
import time
import gc

gc.collect()
question = 'มวยไทยคืออะไร'  # 泰拳是什么
prompt = f'User: {question}\nAssistant:'

inputs = tokenizer(prompt, return_tensors='ms')
input_ids = inputs.input_ids

print(f'输入形状: {input_ids.shape}')
print('开始生成...')
start = time.time()

output = model.generate(
    input_ids,
    max_new_tokens=64,
    do_sample=False,
    num_beams=1,
    eos_token_id=tokenizer.eos_token_id
)

elapsed = time.time() - start
new_tokens = output[0][input_ids.shape[1]:].tolist()
response = tokenizer.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True)

print(f'\n✓ 推理完成！耗时: {elapsed:.2f}秒')
print(f'问题: {question}')
print(f'回答: {response}')


输入形状: (1, 8)
开始生成...
..
✓ 推理完成！耗时: 67.56秒
问题: มวยไทยคืออะไร
回答: มวยไทยเป็นศิลปะการป้องกันตัวที่ใช้ร่างกายทั้งตัวและศีรษะปกป้องร่างกายจากอันตรายหรือการบาดเจ็บ โดยมักใช้ศอก เข่า เท้า และท่าศอก-เข่า


## 多模态RAG推理

RAG流程：用户提问 → 检索SFT知识库+ 图文知识库→ 注入Prompt → 模型生成回答 → 界面同时展示文字答案和相关图片。

In [13]:
from mindnlp.transformers import TextIteratorStreamer
from threading import Thread
import gc

def rag_stream(query, history, use_rag=True):
    retrieved_text = retrieve_sft(query, top_k=1) if use_rag else []
    retrieved_img  = retrieve_images(query, top_k=2) if use_rag else []

    # 构建 context
    context = ''
    if retrieved_text:
        r = retrieved_text[0]
        context = f"ความรู้อ้างอิง:\n{r['prompt'][:100]}\n{r['answer'][:150]}\n\n"

    # 按照 wongwian 的 chat template 格式手动构建 prompt
    prompt = f"{context}User: {query}\nAssistant:"

    gc.collect()  # 推理前清理内存

    inputs = tokenizer(prompt, return_tensors='ms')
    input_ids = inputs.input_ids
    
    streamer = TextIteratorStreamer(
        tokenizer, timeout=120, skip_prompt=True, skip_special_tokens=True
    )
    t = Thread(target=model.generate, kwargs=dict(
        input_ids=input_ids, streamer=streamer,
        max_new_tokens=64,
        do_sample=False,
        num_beams=1,
        eos_token_id=tokenizer.eos_token_id
    ))
    t.start()

    partial = ''
    for token in streamer:
        partial += token
        yield partial.strip(), retrieved_img
    
    # 在回答后面添加图片链接或提示
    if retrieved_img:
        partial += '\n\n相关图片链接：'
        for i, img in enumerate(retrieved_img[:3], 1):
            url = img.get('url', '') if isinstance(img, dict) else img
            partial += f'\n图片{i}: {url}'
    else:
        partial += '\n\n未找到相关图片'
    
    yield partial.strip(), retrieved_img
    
    # 推理完成后清理
    del inputs, input_ids
    gc.collect()

print('RAG推理函数定义完成')

RAG推理函数定义完成


## 视频检索模块

对视频元数据构建TF-IDF索引，支持关键词检索和标签筛选，返回视频摘要和YouTube链接。

In [ ]:
def search_video(query, label_filter='', top_k=5):
    # 视频用关键词匹配，不向量化，省内存
    if not query.strip():
        results = [v for v in video_list if not label_filter or label_filter in v['level2']]
        return results[:top_k]
    query_lower = query.lower()
    results = []
    for v in video_list:
        if label_filter and label_filter not in v['level2']:
            continue
        if query_lower in v['summary'].lower():
            results.append({**v, 'score': 1.0})
            if len(results) >= top_k:
                break
    return results

# 测试
test_results = search_video('มวยไทย', top_k=3)
print(f'视频检索测试: {len(test_results)} 条')
for r in test_results:
    print(f'  {r["url"]}')

视频检索测试: 0 条


## 启动Gradio多模态交互界面

三个Tab：
- **多模态RAG问答**：文字回答 + 相关图片展示
- **图文文化浏览**：按标签分类浏览泰国文化图片
- **视频内容检索**：关键词/标签检索泰语视频

在浏览器打开 [http://127.0.0.1:8090](http://127.0.0.1:8090) 开始使用

In [15]:
import gradio as gr

all_img_labels = sorted(set(
    lv2
    for d in image_data
    for lv2 in d.get('labels', {}).get('pjwk_cates', {}).get('level2', [])
))
all_video_labels = sorted(v_label_counter.keys())

def browse_images(label, page):
    filtered = [
        d for d in image_data
        if label in d.get('labels', {}).get('pjwk_cates', {}).get('level2', [])
    ]
    total = len(filtered)
    start = int(page) * 9
    end = min(start + 9, total)
    gallery = [(d['image']['path'], d['captions']['content']) for d in filtered[start:end]]
    info = f'标签「{label}」共 {total} 张，第 {int(page)+1} 页（{start+1}-{end}）'
    return gallery, info

def search_videos(query, label):
    label_filter = '' if label == '全部' else label
    results = search_video(query, label_filter=label_filter, top_k=8)
    if not results:
        return '<p>未找到相关视频</p>'
    html = '<div style="display:flex;flex-wrap:wrap;gap:12px;">'
    for r in results:
        lv2 = ', '.join(r['level2']) if r['level2'] else '未知'
        summary = r['summary'][:120] + '...' if len(r['summary']) > 120 else r['summary']
        html += (
            '<div style="border:1px solid #ddd;border-radius:8px;padding:12px;width:280px;">'
            f'<div style="font-size:12px;color:#888;margin-bottom:6px;">🏷️ {lv2}</div>'
            f'<div style="font-size:13px;margin-bottom:8px;">{summary}</div>'
            f'<a href="{r["url"]}" target="_blank" style="color:#1a73e8;font-size:12px;">▶ 观看视频</a>'
            '</div>'
        )
    html += '</div>'
    return html

with gr.Blocks(title='泰国文化多模态探索平台', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🇹🇭 泰国文化多模态探索平台')
    gr.Markdown('基于万卷·丝路泰语数据集| wongwian-micro-instruct + MindSpore')

    with gr.Tab('多模态RAG问答'):
        gr.Markdown('输入泰语或中文问题，系统从知识库检索相关内容并附带相关图片。')
        with gr.Row():
            with gr.Column(scale=3):
                chatbot = gr.Chatbot(height=400, label='对话')
                msg_input = gr.Textbox(
                    placeholder='输入问题，如：มวยไทยคืออะไร',
                    label='问题'
                )
                with gr.Row():
                    submit_btn = gr.Button('发送（RAG）', variant='primary')
                    no_rag_btn = gr.Button('发送（无RAG对比）')
                    clear_btn  = gr.Button('清空')
                gr.Examples(
                    examples=[
                        'กีฬาประจำชาติของไทยคืออะไร',
                        'ต้มยำคืออะไร',
                        'ประเพณีสงกรานต์คืออะไร',
                        'ผัดไทยคืออะไร',
                    ],
                    inputs=msg_input
                )
            with gr.Column(scale=2):
                ref_gallery = gr.Gallery(label='检索到的相关图片', columns=3, height=400)

        def user_submit(message, history):
            return '', history + [[message, '']]

        def bot_rag(history):
            message = history[-1][0]
            prev = history[:-1]
            img_urls = []
            for partial, imgs in rag_stream(message, prev, use_rag=True):
                history[-1][1] = partial
                img_urls = imgs
                yield history, [(url, '') for url in img_urls[:3]]

        def bot_no_rag(history):
            message = history[-1][0]
            prev = history[:-1]
            for partial, _ in rag_stream(message, prev, use_rag=False):
                history[-1][1] = partial
                yield history, []

        submit_btn.click(user_submit, [msg_input, chatbot], [msg_input, chatbot]).then(
            bot_rag, chatbot, [chatbot, ref_gallery]
        )
        no_rag_btn.click(user_submit, [msg_input, chatbot], [msg_input, chatbot]).then(
            bot_no_rag, chatbot, [chatbot, ref_gallery]
        )
        clear_btn.click(lambda: ([], []), outputs=[chatbot, ref_gallery])

    with gr.Tab('图文文化浏览'):
        gr.Markdown('按标签分类浏览泰国文化图片，展示泰语描述。')
        with gr.Row():
            img_label_dd = gr.Dropdown(
                choices=all_img_labels,
                value=all_img_labels[0] if all_img_labels else '',
                label='选择标签'
            )
            img_page_slider = gr.Slider(0, 50, value=0, step=1, label='页码')
        img_info    = gr.Textbox(label='当前页信息', interactive=False)
        img_gallery = gr.Gallery(label='图片', columns=3, height=500)

        img_label_dd.change(browse_images, [img_label_dd, img_page_slider], [img_gallery, img_info])
        img_page_slider.change(browse_images, [img_label_dd, img_page_slider], [img_gallery, img_info])
        demo.load(browse_images, [img_label_dd, img_page_slider], [img_gallery, img_info])

    with gr.Tab('视频内容'):
        gr.Markdown('输入泰语关键词或选择标签，检索相关泰语视频，点击链接跳转YouTube。')
        with gr.Row():
            video_query    = gr.Textbox(placeholder='输入泰语关键词，如：มวยไทย', label='关键词（可留空）')
            video_label_dd = gr.Dropdown(choices=['全部'] + all_video_labels, value='全部', label='标签筛选')
            video_btn      = gr.Button('搜索', variant='primary')
        video_results = gr.HTML(label='搜索结果')
        video_btn.click(search_videos, [video_query, video_label_dd], video_results)
        gr.Examples(
            examples=[['มวยไทย', '全部'], ['อาหาร', '全部'], ['', 'culture']],
            inputs=[video_query, video_label_dd]
        )

demo.launch(server_name='0.0.0.0', server_port=8090)

/home/HwHiAiUser/.local/lib/python3.9/site-packages/gradio/components/dropdown.py:188: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: culture or set allow_custom_value=True.
  warnings.warn(


Running on local URL:  http://0.0.0.0:8090

To create a public link, set `share=True` in `launch()`.


/home/HwHiAiUser/.local/lib/python3.9/site-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


..

## 总结

本项目在香橙派AIpro开发板上实现了泰国文化多模态探索平台，完整利用了万卷·丝路泰语数据集的三种模态：

- **SFT数据**：构建文字知识库，支持RAG检索增强问答
- **图文数据**：构建图片知识库，RAG回答时附带相关图片，真正体现多模态
- **视频数据**：构建视频检索索引，支持关键词+标签双重筛选

硬件：昇腾AI开发板
开发板镜像: Ubuntu镜像
CANN 8.1RC1；MindSpore: 2.5.0或mindspore2.6；8.1RC1beta1
MindNLP: 0.4.1
Python: 3.9